# 01 · Ingest sample AMR CSV → Lakehouse table `isolates`

In [1]:
for k, v in spark.sparkContext.getConf().getAll():
    if "AZOPENAI" in k.upper():
        print(k, "=", v)
# or individually:
print("endpoint:", spark.conf.get("spark.azopenai.endpoint", None))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 3, Finished, Available, Finished)

endpoint: https://20190-mhdrlnmt-eastus2.cognitiveservices.azure.com/


In [2]:
# 01 — Clean ingest for Fabric → overwrite Delta table `isolates`

from notebookutils import mssparkutils
from pyspark.sql import functions as F

# Prefer the canonical Fabric path; fall back to Files/ if needed
candidates = [
    "Files/fabric/data/sample/sample_amr.csv",
    "Files/sample_amr.csv",
]
existing = [p for p in candidates if mssparkutils.fs.exists(p)]
if not existing:
    raise FileNotFoundError(
        f"No sample CSV found. Expected one of: {candidates}"
    )

read_path = existing[0]
print("Using input:", read_path)

# Load and normalize schema
df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(read_path)
        .select(
            "province","organism","antibiotic","specimen","sector",
            "year","n_tested","percent_resistant"
        )
        .withColumn("province", F.trim(F.col("province")))
        .withColumn("organism", F.trim(F.col("organism")))
        .withColumn("antibiotic", F.trim(F.col("antibiotic")))
        .withColumn("specimen", F.initcap(F.trim(F.col("specimen"))))  # e.g., Blood/Urine
        .withColumn("sector",   F.initcap(F.trim(F.col("sector"))))    # Public
        .withColumn("year", F.col("year").cast("int"))
        .withColumn("n_tested", F.col("n_tested").cast("int"))
        .withColumn("percent_resistant", F.col("percent_resistant").cast("double"))
)

# Basic sanity checks (raise early if something is off)
bad = df.where(
    F.col("province").isNull() | F.col("organism").isNull() | F.col("antibiotic").isNull() |
    F.col("specimen").isNull() | F.col("sector").isNull() |
    (F.col("year") < 2000) | (F.col("year") > 2100) |
    (F.col("n_tested") <= 0) |
    (F.col("percent_resistant") < 0) | (F.col("percent_resistant") > 100)
)
if bad.limit(1).count() > 0:
    display(bad.limit(20))
    raise ValueError("Data validation failed; see sample bad rows above.")

# Write managed Delta table
spark.sql("DROP TABLE IF EXISTS isolates")
df.write.mode("overwrite").format("delta").saveAsTable("isolates")
spark.sql("REFRESH TABLE isolates")

print("Rows in isolates:", spark.table("isolates").count())
display(spark.table("isolates").orderBy("province","organism","antibiotic","year").limit(20))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 4, Finished, Available, Finished)

Using input: Files/fabric/data/sample/sample_amr.csv
Rows in isolates: 2000


SynapseWidget(Synapse.DataFrame, 81297bc8-8566-493b-9eda-6c6bb1ba1f93)

In [3]:
# 02 — EB shrinkage, τ grid search, stability → write Delta tables: thresholds, alerts

from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import DoubleType

iso = spark.table("isolates")

# resistant counts from percent
iso = iso.withColumn("r", F.round(F.col("percent_resistant")/100.0 * F.col("n_tested")).cast("int"))
iso.cache(); iso.count()

# Beta–Binomial posterior (Bayes–Laplace, +1/+1)
eb = (iso
  .select("province","organism","antibiotic","specimen","year","n_tested","r")
  .withColumn("alpha", F.col("r") + F.lit(1.0))
  .withColumn("beta",  (F.col("n_tested") - F.col("r")) + F.lit(1.0))
  .withColumn("theta_hat", (F.col("alpha"))/(F.col("alpha")+F.col("beta")))
)

# UDF for Pr(theta > tau) with safe fallbacks
def _pr_exceed(a,b,tau):
    try:
        from scipy.special import betainc
        return float(1.0 - betainc(a,b,tau))
    except Exception:
        try:
            import mpmath as mp
            return float(1.0 - mp.betainc(a,b, 0, tau, regularized=True))
        except Exception:
            # last-resort heuristic
            return 1.0 if (a/(a+b)) > tau else 0.0

pr_exceed = F.udf(_pr_exceed, DoubleType())

# τ grid 0.10–0.40 (step 0.01)
taus = [round(x/100,2) for x in range(10,41)]
tau_df = spark.createDataFrame([(t,) for t in taus], ["tau"])

# score taus on latest year per (organism,antibiotic,specimen)
latest = eb.agg(F.max("year").alias("y")).collect()[0]["y"]
crit = (eb.where(F.col("year")==latest)
          .crossJoin(tau_df)
          .withColumn("pr_exceed_tau", pr_exceed("alpha","beta","tau"))
          .groupBy("organism","antibiotic","specimen","tau")
          .agg(F.avg("pr_exceed_tau").alias("score")))

w = W.partitionBy("organism","antibiotic","specimen").orderBy(F.col("score").desc(), F.col("tau").asc())
thresholds = (crit
  .withColumn("rank", F.row_number().over(w))
  .where("rank=1")
  .select("organism","antibiotic","specimen",
          F.lit(latest).alias("year_ref"),
          "tau",
          F.lit("grid_search_latest_year").alias("method"))
)

# attach τ, compute gate + persistence + 3-yr slope, impact, reason
alerts = (eb
  .join(thresholds.select("organism","antibiotic","specimen","tau"),
        ["organism","antibiotic","specimen"], "left")
  .withColumn("pr_exceed_tau", pr_exceed("alpha","beta","tau"))
  .withColumn("gate", F.col("pr_exceed_tau") >= F.lit(0.80))
)

w_year = W.partitionBy("province","organism","antibiotic","specimen").orderBy("year")
alerts = (alerts
  .withColumn("gate_prev", F.lag("gate").over(w_year))
  .withColumn("persistence", F.when(F.col("gate") & F.col("gate_prev"), True).otherwise(False))
  .withColumn("pr_prev2", F.lag("pr_exceed_tau",2).over(w_year))
  .withColumn("slope3", (F.col("pr_exceed_tau") - F.col("pr_prev2"))/F.lit(2.0))
  .withColumn("slope_ok", F.col("slope3") >= F.lit(0.05))
  .withColumn("is_stable_alert", F.col("gate") & (F.col("persistence") | F.col("slope_ok")))
  .withColumn("impact_score", F.when(F.col("theta_hat") > F.col("tau"),
                                    (F.col("theta_hat") - F.col("tau"))*F.lit(1000.0)).otherwise(F.lit(0.0)))
  .withColumn("reason",
              F.when(F.col("is_stable_alert"),
                     F.when(F.col("persistence"), F.lit("gate+persistence"))
                      .otherwise(F.lit("gate+trend")))
               .otherwise(F.lit("no_stable_rule")))
)

# Write Delta tables
spark.sql("DROP TABLE IF EXISTS thresholds")
thresholds.write.mode("overwrite").format("delta").saveAsTable("thresholds")
spark.sql("REFRESH TABLE thresholds")

spark.sql("DROP TABLE IF EXISTS alerts")
(alerts
  .select("province","organism","antibiotic","specimen","year",
          "n_tested","theta_hat","tau","pr_exceed_tau",
          "is_stable_alert","impact_score","reason")
  .write.mode("overwrite").format("delta").saveAsTable("alerts"))
spark.sql("REFRESH TABLE alerts")

print("thresholds rows:", spark.table("thresholds").count())
print("alerts rows:", spark.table("alerts").count())
display(spark.table("thresholds").orderBy("organism","antibiotic","specimen"))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 5, Finished, Available, Finished)

thresholds rows: 80
alerts rows: 2000


SynapseWidget(Synapse.DataFrame, 34f27426-0379-417e-88d0-8710da129a71)

In [4]:
# --- Clean reset: drop helper objects safely, then recreate views

names = [
    "alerts_latest", "thresholds_latest", "alerts_trend_3yr",
    "v_alerts", "v_thresholds", "v_isolates"
]

# 1) Drop VIEW first, then TABLE (Spark errors if you try DROP TABLE on a view)
for n in names:
    spark.sql(f"DROP VIEW IF EXISTS {n}")
    spark.sql(f"DROP TABLE IF EXISTS {n}")

# 2) Recreate views
spark.sql("""
CREATE VIEW alerts_latest AS
SELECT * FROM (
  SELECT a.*,
         ROW_NUMBER() OVER (
           PARTITION BY province, organism, antibiotic, specimen
           ORDER BY year DESC
         ) rn
  FROM alerts a
) WHERE rn = 1
""")

spark.sql("""
CREATE VIEW thresholds_latest AS
SELECT * FROM (
  SELECT t.*,
         ROW_NUMBER() OVER (
           PARTITION BY organism, antibiotic, specimen
           ORDER BY year_ref DESC
         ) rn
  FROM thresholds t
) WHERE rn = 1
""")

spark.sql("""
CREATE VIEW alerts_trend_3yr AS
SELECT province, organism, antibiotic, specimen, year, pr_exceed_tau
FROM (
  SELECT a.*,
         MAX(year) OVER (
           PARTITION BY province, organism, antibiotic, specimen
         ) AS y_max
  FROM alerts a
)
WHERE year >= y_max - 2
""")

spark.sql("CREATE VIEW v_alerts      AS SELECT * FROM alerts")
spark.sql("CREATE VIEW v_thresholds  AS SELECT * FROM thresholds")
spark.sql("CREATE VIEW v_isolates    AS SELECT * FROM isolates")

# 3) Smoke test
display(spark.sql("""
SELECT province, tau, theta_hat, pr_exceed_tau, is_stable_alert, reason, n_tested, year
FROM v_alerts
WHERE organism='E. coli'
  AND antibiotic='Ciprofloxacin'
  AND year=2024
  AND is_stable_alert = TRUE
ORDER BY province
"""))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, efedac8d-bd7a-4681-8c5f-ebdaecaeabbf)

In [5]:
import os, time, json, requests
from pyspark.sql import functions as F, types as T
from datetime import datetime

def _get(key, required=True):
    v = spark.conf.get(f"spark.azopenai.{key}", None) or os.environ.get(f"AZURE_OPENAI_{key.upper()}")
    if required and not v:
        raise RuntimeError(f"Missing config for '{key}'.")
    return v

AZ_ENDPOINT   = _get("endpoint").rstrip("/")
AZ_DEPLOYMENT = _get("deployment")
AZ_API_VER    = _get("api_version")
AZ_API_KEY    = _get("api_key")

CHAT_URL = f"{AZ_ENDPOINT}/openai/deployments/{AZ_DEPLOYMENT}/chat/completions?api-version={AZ_API_VER}"
HEADERS  = {"api-key": AZ_API_KEY, "Content-Type": "application/json"}

def call_aoai(system, user, max_tokens=220, temperature=0.1, retries=4):
    payload = {
        "messages": [{"role":"system","content":system},{"role":"user","content":user}],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "response_format": {"type":"json_object"}
    }
    delay = 1.5
    for _ in range(retries):
        r = requests.post(CHAT_URL, headers=HEADERS, json=payload, timeout=60)
        if r.status_code == 200:
            return r.json()["choices"][0]["message"]["content"]
        if r.status_code in (429, 500, 503):
            time.sleep(delay); delay *= 1.8; continue
        raise RuntimeError(f"AOAI error {r.status_code}: {r.text[:300]}")
    raise RuntimeError("AOAI rate/availability error after retries")

# ---- Select facts to summarize
alerts = spark.sql("""
SELECT province, organism, antibiotic, specimen, year,
       tau, theta_hat, pr_exceed_tau, is_stable_alert, reason, n_tested
FROM alerts
WHERE organism='E. coli' AND antibiotic='Ciprofloxacin' AND year=2024
ORDER BY province
""").collect()

system_prompt = (
  'You are a cautious policy analyst. Return JSON with keys '
  '{"title": string, "bullets": [string, string, string], "sms": string}. '
  'Use only the provided numbers; sms ≤ 160 chars.'
)

rows = []
for r in alerts:
    user_prompt = (
        f"Province: {r['province']}\nOrganism: {r['organism']}\nAntibiotic: {r['antibiotic']}\n"
        f"Specimen: {r['specimen']}\nYear: {r['year']}\n"
        f"tau: {float(r['tau']):.2f}\n"
        f"theta_hat: {float(r['theta_hat']):.3f}\n"
        f"Pr(theta>tau): {float(r['pr_exceed_tau']):.3f}\n"
        f"Stable alert?: {bool(r['is_stable_alert'])}\nReason: {r['reason']}\n"
        f"n_tested: {int(r['n_tested'])}\n\n"
        "Write a 3–5 sentence brief:\n"
        "1) Status quoting tau and Pr(theta>tau).\n"
        "2) Which rule fired (Gate+Persistence or Gate+Slope).\n"
        "3) Practical note (sample size/impact).\n"
        "4) One action line.\n"
        "Return ONLY valid JSON per schema."
    )
    try:
        parsed = json.loads(call_aoai(system_prompt, user_prompt))
        title   = parsed.get("title") or f"{r['province']}: {r['organism']}–{r['antibiotic']} ({r['year']})"
        bullets = parsed.get("bullets") or []
        sms     = parsed.get("sms") or ""
    except Exception:
        status  = "Stable alert" if r['is_stable_alert'] else "No stable alert"
        title   = f"{r['province']}: {r['organism']}–{r['antibiotic']} ({r['year']}) — {status}"
        bullets = [
            f"τ={float(r['tau']):.2f}, θ̂={float(r['theta_hat']):.2f}, Pr(θ>τ)={float(r['pr_exceed_tau']):.2f}",
            f"Rule: {r['reason']}; n={int(r['n_tested'])}",
            "Action: Review empiric policy." if r['is_stable_alert'] else "Action: Continue monitoring."
        ]
        sms = (f"{r['province']} {r['organism']}/{r['antibiotic']} {r['year']}: {status} "
               f"(Pr>{float(r['tau']):.2f}={float(r['pr_exceed_tau']):.2f}, n={int(r['n_tested'])}).")[:160]

    rows.append({
        "row_key": "|".join([r['province'], r['organism'], r['antibiotic'], r['specimen'], str(r['year'])]),
        "title": title, "bullets": bullets, "sms": sms, "created_at": datetime.utcnow()
    })

schema = T.StructType([
    T.StructField("row_key", T.StringType(), False),
    T.StructField("title", T.StringType(), False),
    T.StructField("bullets", T.ArrayType(T.StringType()), False),
    T.StructField("sms", T.StringType(), False),
    T.StructField("created_at", T.TimestampType(), True),
])

spark.sql("DROP TABLE IF EXISTS alerts_ai_summaries")
spark.createDataFrame(rows, schema).write.mode("overwrite").format("delta").saveAsTable("alerts_ai_summaries")
spark.sql("REFRESH TABLE alerts_ai_summaries")
display(spark.table("alerts_ai_summaries").orderBy("row_key").limit(20))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 00dc6e41-5f4f-440c-abc0-13d8fd4acada)

In [6]:
%%sql
-- 1) Count + distinct provinces
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT element_at(split(row_key, '\\|'), 1)) AS provinces
FROM alerts_ai_summaries;

%%sql
-- 2) Make sure SMS is within 160 chars
SELECT row_key, LENGTH(sms) AS sms_len
FROM alerts_ai_summaries
ORDER BY sms_len DESC
LIMIT 5;

%%sql
-- 3) Peek one record fully
SELECT *
FROM alerts_ai_summaries
WHERE row_key LIKE 'Gauteng|E. coli|Ciprofloxacin%'
LIMIT 1;


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 10, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 5 rows and 2 fields>

<Spark SQL result set with 1 rows and 5 fields>

In [7]:
# --- Agent tool view: alerts_with_brief (alerts JOIN alerts_ai_summaries by row_key) ---
from pyspark.sql import functions as F

# deterministic row_key builder (must match how 03 wrote summaries)
spark.sql("DROP VIEW IF EXISTS alerts_with_brief")

spark.sql("""
CREATE VIEW alerts_with_brief AS
WITH base AS (
  SELECT
    a.*,
    CONCAT_WS('|', a.province, a.organism, a.antibiotic, a.specimen, CAST(a.year AS STRING)) AS row_key
  FROM alerts a
)
SELECT
  b.province, b.organism, b.antibiotic, b.specimen, b.year,
  b.theta_hat, b.tau, b.pr_exceed_tau, b.is_stable_alert, b.impact_score, b.reason, b.n_tested,
  s.title, s.bullets, s.sms
FROM base b
LEFT JOIN alerts_ai_summaries s
  ON b.row_key = s.row_key
""")

# -- Smoke test: E. coli / Ciprofloxacin / 2024 with brief
display(spark.sql("""
SELECT province, tau, pr_exceed_tau, is_stable_alert, reason, n_tested, title, bullets, sms
FROM alerts_with_brief
WHERE organism='E. coli' AND antibiotic='Ciprofloxacin' AND year=2024
ORDER BY province
"""))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, eea8ebdd-49a0-44a4-a464-813e8a03722c)

In [8]:
stmts = [
"""
CREATE OR REPLACE VIEW alerts_with_brief AS
WITH base AS (
  SELECT
      a.*,
      CONCAT_WS('|', a.province, a.organism, a.antibiotic, a.specimen, CAST(a.year AS STRING)) AS row_key
  FROM alerts a
)
SELECT b.*, s.title, s.bullets, s.sms
FROM base b
LEFT JOIN alerts_ai_summaries s USING (row_key)
ORDER BY b.province
""",
"""
CREATE OR REPLACE VIEW thresholds_latest AS
SELECT *
FROM (
  SELECT
      t.*,
      ROW_NUMBER() OVER (
        PARTITION BY organism, antibiotic, specimen
        ORDER BY year_ref DESC
      ) AS rn
  FROM thresholds t
) WHERE rn = 1
""",
"""
CREATE OR REPLACE VIEW alerts_trend_3yr AS
SELECT province, organism, antibiotic, specimen, year, pr_exceed_tau
FROM (
  SELECT
      a.*,
      MAX(year) OVER (PARTITION BY province, organism, antibiotic, specimen) AS y_max
  FROM alerts a
)
WHERE year >= y_max - 2
"""
]

for s in stmts:
    spark.sql(s)
print("Views created/refreshed: alerts_with_brief, thresholds_latest, alerts_trend_3yr")


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 12, Finished, Available, Finished)

Views created/refreshed: alerts_with_brief, thresholds_latest, alerts_trend_3yr


In [9]:
display(spark.sql("SELECT * FROM thresholds_latest ORDER BY organism, antibiotic, specimen"))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 306adcad-0a0c-41e5-b4a6-17ea0d21c7a5)

In [10]:
display(spark.sql("""
SELECT province, tau, pr_exceed_tau, is_stable_alert, reason, title, bullets, sms
FROM alerts_with_brief
WHERE organism='E. coli' AND antibiotic='Ciprofloxacin' AND year=2024
ORDER BY province
"""))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a9f9b175-b544-42e3-9aa6-533370d356f4)

In [11]:
# Materialize reporting tables from the views
stmts = [
    # 1) alerts_with_brief_tbl
    "DROP TABLE IF EXISTS alerts_with_brief_tbl",
    "CREATE TABLE alerts_with_brief_tbl USING DELTA AS SELECT * FROM alerts_with_brief",

    # 2) thresholds_latest_tbl
    "DROP TABLE IF EXISTS thresholds_latest_tbl",
    "CREATE TABLE thresholds_latest_tbl USING DELTA AS SELECT * FROM thresholds_latest",

    # 3) alerts_trend_3yr_tbl
    "DROP TABLE IF EXISTS alerts_trend_3yr_tbl",
    "CREATE TABLE alerts_trend_3yr_tbl USING DELTA AS SELECT * FROM alerts_trend_3yr",
]

for s in stmts:
    spark.sql(s)

print("Materialized: alerts_with_brief_tbl, thresholds_latest_tbl, alerts_trend_3yr_tbl")


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 15, Finished, Available, Finished)

Materialized: alerts_with_brief_tbl, thresholds_latest_tbl, alerts_trend_3yr_tbl


In [12]:
display(spark.sql("SELECT * FROM alerts_with_brief_tbl LIMIT 5"))
display(spark.sql("SELECT * FROM thresholds_latest_tbl  LIMIT 5"))
display(spark.sql("SELECT * FROM alerts_trend_3yr_tbl LIMIT 5"))


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 16, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f9a44313-d50c-43c1-a4ca-0d978f886c6d)

SynapseWidget(Synapse.DataFrame, e2e61afd-a2d0-496a-a306-305d4a81a224)

SynapseWidget(Synapse.DataFrame, c833448f-57ce-499c-bc96-77e331b9ff46)

## Demo / Fallback SQL (no Agent)


In [13]:
organism   = "E. coli"
antibiotic = "Ciprofloxacin"
year       = 2024

q1 = f"""
SELECT a.province,
       COALESCE(a.tau, tl.tau) AS tau,
       a.pr_exceed_tau,
       a.is_stable_alert,
       a.reason
FROM alerts a
LEFT JOIN thresholds_latest tl
  ON tl.organism = a.organism
 AND tl.antibiotic = a.antibiotic
 AND tl.specimen  = a.specimen
WHERE a.organism   = '{organism}'
  AND a.antibiotic = '{antibiotic}'
  AND a.year       = {year}
  AND a.is_stable_alert = 1
ORDER BY a.province;
"""
df1 = spark.sql(q1)
display(df1)


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, fbf03e4a-f630-4737-85f5-0085efe10601)

In [14]:
province = "Limpopo"

q2 = f"""
SELECT a.province, a.organism, a.antibiotic, a.specimen, a.year,
       COALESCE(a.tau, tl.tau) AS tau,
       a.pr_exceed_tau, a.is_stable_alert, a.reason,
       b.title, b.bullets, b.sms
FROM alerts a
LEFT JOIN thresholds_latest tl
  ON tl.organism = a.organism
 AND tl.antibiotic = a.antibiotic
 AND tl.specimen  = a.specimen
LEFT JOIN alerts_with_brief b
  ON b.province   = a.province
 AND b.organism   = a.organism
 AND b.antibiotic = a.antibiotic
 AND b.specimen   = a.specimen
 AND b.year       = a.year
WHERE a.province   = '{province}'
  AND a.organism   = '{organism}'
  AND a.antibiotic = '{antibiotic}'
  AND a.year       = {year};
"""
df2 = spark.sql(q2)
display(df2)


StatementMeta(, a43e2bbf-99d4-4be8-91a9-783875d58369, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 028cf54a-6539-46cd-893d-21f1f0dd6627)